In [1]:
"""
Multivariable Logistic Regression: Odds Ratio Analysis
- Outcome: PD-NC vs PD-CI (PD-MCI + PD-D)
- Each biomarker as predictor, controlling for sex, age, education, disease duration
- Combined cohort + individual cohorts (JHU, UW)

Upload '02092026_cross_sectional.csv' to Colab before running.
"""

# ============================================================
# 1. Setup
# ============================================================
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# pip install statsmodels (Colab already has it)
import statsmodels.api as sm
from statsmodels.formula.api import logit

# ============================================================
# 2. Load Data
# ============================================================
# Colab: upload file first, then adjust path
df = pd.read_csv('/content/02092026_cross_sectional.csv')

print(f"Total samples: {len(df)}")
print(f"Labels: {df['label'].value_counts().to_dict()}")
print(f"Cohorts: {df['Sources'].value_counts().to_dict()}")

# ============================================================
# 3. Prepare Data - PD only (exclude HC), create binary outcome
# ============================================================
df_pd = df[df['label'].isin(['PD-NC', 'PD-MCI', 'PD-D'])].copy()
df_pd['outcome'] = (df_pd['label'].isin(['PD-MCI', 'PD-D'])).astype(int)  # 0=PD-NC, 1=PD-CI

print(f"\nPD patients only: {len(df_pd)}")
print(f"PD-NC (outcome=0): {(df_pd['outcome']==0).sum()}")
print(f"PD-CI (outcome=1): {(df_pd['outcome']==1).sum()}")

# ============================================================
# 4. Define biomarkers and covariates
# ============================================================
biomarkers = [
    'tht-mfi', 'tht-tlag', 'tht-T20', 'tht-t50', 'forming rate',
    'dls-peak-number', 'dls-peak-1-intensity', 'dls-peak-2-intensity',
    'dls-peak-1-size', 'dls-peak-2-size', 'dls-peak-1-pd', 'dls-peak-2-pd',
    'neurotoxicity'
]

covariates = ['sex', 'age', 'education', 'disease duration']

# ============================================================
# 5. Run Multivariable Logistic Regression
# ============================================================
def run_logistic_regression(data, biomarker, covariates, cohort_name="Combined"):
    """
    Run logistic regression: outcome ~ biomarker + covariates
    Returns dict with odds ratios, CIs, p-values for all variables
    """
    # Prepare predictors
    predictors = [biomarker] + covariates
    X = data[predictors].copy()
    y = data['outcome'].copy()

    # Drop rows with missing values
    mask = X.notna().all(axis=1) & y.notna()
    X = X[mask]
    y = y[mask]

    if len(y) < 10 or y.nunique() < 2:
        return None

    # Add constant
    X = sm.add_constant(X)

    try:
        model = sm.Logit(y, X).fit(disp=0, maxiter=100)

        results = []
        for var in predictors:
            idx = list(X.columns).index(var)
            coef = model.params.iloc[idx]
            se = model.bse.iloc[idx]
            z = model.tvalues.iloc[idx]
            p = model.pvalues.iloc[idx]
            ci_low, ci_high = model.conf_int().iloc[idx]

            results.append({
                'Cohort': cohort_name,
                'Biomarker_Model': biomarker,
                'Variable': var,
                'Odds_Ratio': np.exp(coef),
                'Std_Err': se,
                'z': z,
                'P_value': p,
                'CI_lower': np.exp(ci_low),
                'CI_upper': np.exp(ci_high),
                'N': len(y)
            })

        # Add intercept
        idx_const = list(X.columns).index('const')
        results.append({
            'Cohort': cohort_name,
            'Biomarker_Model': biomarker,
            'Variable': '_cons',
            'Odds_Ratio': np.exp(model.params.iloc[idx_const]),
            'Std_Err': model.bse.iloc[idx_const],
            'z': model.tvalues.iloc[idx_const],
            'P_value': model.pvalues.iloc[idx_const],
            'CI_lower': np.exp(model.conf_int().iloc[idx_const, 0]),
            'CI_upper': np.exp(model.conf_int().iloc[idx_const, 1]),
            'N': len(y)
        })

        return results

    except Exception as e:
        print(f"  Error for {biomarker} ({cohort_name}): {e}")
        return None

# ============================================================
# 6. Run for all biomarkers x all cohorts
# ============================================================
all_results = []

cohort_configs = [
    ("Combined", df_pd),
    ("JHU", df_pd[df_pd['Sources'] == 'JHU']),
    ("UW", df_pd[df_pd['Sources'] == 'UW']),
]

for cohort_name, cohort_data in cohort_configs:
    print(f"\n{'='*60}")
    print(f"Cohort: {cohort_name} (n={len(cohort_data)}, NC={sum(cohort_data['outcome']==0)}, CI={sum(cohort_data['outcome']==1)})")
    print(f"{'='*60}")

    for biomarker in biomarkers:
        res = run_logistic_regression(cohort_data, biomarker, covariates, cohort_name)
        if res:
            all_results.extend(res)
            # Print key result (biomarker OR only)
            bio_res = [r for r in res if r['Variable'] == biomarker][0]
            sig = '***' if bio_res['P_value'] < 0.001 else '**' if bio_res['P_value'] < 0.01 else '*' if bio_res['P_value'] < 0.05 else ''
            print(f"  {biomarker:25s} OR={bio_res['Odds_Ratio']:.4f}  "
                  f"95%CI ({bio_res['CI_lower']:.4f}-{bio_res['CI_upper']:.4f})  "
                  f"p={bio_res['P_value']:.4f} {sig}")

# ============================================================
# 7. Create Results DataFrame
# ============================================================
results_df = pd.DataFrame(all_results)

# Format for display
def format_results(df):
    df = df.copy()
    df['OR_formatted'] = df['Odds_Ratio'].apply(lambda x: f"{x:.4f}" if x < 1000 else f"{x:.2e}")
    df['CI_formatted'] = df.apply(lambda r: f"({r['CI_lower']:.4f}-{r['CI_upper']:.4f})"
                                  if r['CI_upper'] < 1000
                                  else f"({r['CI_lower']:.2e}-{r['CI_upper']:.2e})", axis=1)
    df['P_formatted'] = df['P_value'].apply(lambda x: f"{x:.4f}" if x >= 0.001 else f"{x:.2e}")
    df['Sig'] = df['P_value'].apply(lambda x: '***' if x<0.001 else '**' if x<0.01 else '*' if x<0.05 else '.' if x<0.1 else '')
    return df

results_formatted = format_results(results_df)

# ============================================================
# 8. Display Summary Table (Biomarker OR only, excluding covariates)
# ============================================================
print("\n" + "="*80)
print("SUMMARY: Biomarker Odds Ratios (adjusted for sex, age, education, disease duration)")
print("="*80)

for cohort_name in ["Combined", "JHU", "UW"]:
    print(f"\n--- {cohort_name} Cohort ---")
    cohort_bio = results_formatted[
        (results_formatted['Cohort'] == cohort_name) &
        (results_formatted['Variable'] == results_formatted['Biomarker_Model'])
    ][['Variable', 'OR_formatted', 'CI_formatted', 'P_formatted', 'Sig', 'N']]
    print(cohort_bio.to_string(index=False))

# ============================================================
# 9. Save to Excel (one sheet per cohort, Liana's format)
# ============================================================
output_file = 'Logistic_Regression_OddsRatio_Results.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

    # Sheet 1: Summary (biomarker OR only)
    summary_rows = []
    for cohort_name in ["Combined", "JHU", "UW"]:
        for biomarker in biomarkers:
            mask = (results_formatted['Cohort']==cohort_name) & (results_formatted['Biomarker_Model']==biomarker) & (results_formatted['Variable']==biomarker)
            row = results_formatted[mask]
            if len(row) > 0:
                r = row.iloc[0]
                summary_rows.append({
                    'Cohort': cohort_name,
                    'Biomarker': biomarker,
                    'Odds Ratio': r['Odds_Ratio'],
                    '95% CI lower': r['CI_lower'],
                    '95% CI upper': r['CI_upper'],
                    'z': r['z'],
                    'P-value': r['P_value'],
                    'Sig': r['Sig'],
                    'N': r['N']
                })
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)

    # Sheet 2-4: Full results per cohort (Liana's format - all variables per model)
    for cohort_name in ["Combined", "JHU", "UW"]:
        cohort_data = results_formatted[results_formatted['Cohort']==cohort_name]
        export_cols = ['Biomarker_Model', 'Variable', 'Odds_Ratio', 'Std_Err', 'z', 'P_value', 'CI_lower', 'CI_upper', 'N']
        cohort_data[export_cols].to_excel(writer, sheet_name=f'{cohort_name}', index=False)

    # Sheet 5: Full raw results
    results_formatted.to_excel(writer, sheet_name='All_Raw', index=False)

print(f"\nResults saved to: {output_file}")
print("Sheets: Summary, Combined, JHU, UW, All_Raw")

# ============================================================
# 10. Verification: Compare with Liana's known results
# ============================================================
print("\n" + "="*80)
print("VERIFICATION: Compare with Liana's Table B results (Combined cohort)")
print("="*80)
print("Liana's values → Our values")
print()

verification = {
    'tht-mfi':       {'liana_or': 2.147, 'liana_p': 0.000},
    'tht-tlag':      {'liana_or': 0.82,  'liana_p': 0.000},
    'tht-t50':       {'liana_or': 0.83,  'liana_p': 0.000},
    'neurotoxicity': {'liana_or': 0.92,  'liana_p': 0.000},
}

for bio, liana in verification.items():
    our = results_df[(results_df['Cohort']=='Combined') & (results_df['Biomarker_Model']==bio) & (results_df['Variable']==bio)]
    if len(our) > 0:
        r = our.iloc[0]
        match = "✓" if abs(r['Odds_Ratio'] - liana['liana_or']) < 0.05 else "CHECK"
        print(f"  {bio:20s} Liana OR={liana['liana_or']:.3f} → Ours OR={r['Odds_Ratio']:.4f}  {match}")

Total samples: 227
Labels: {'PD-MCI': 98, 'HC': 49, 'PD-NC': 47, 'PD-D': 33}
Cohorts: {'UW': 114, 'JHU': 113}

PD patients only: 178
PD-NC (outcome=0): 47
PD-CI (outcome=1): 131

Cohort: Combined (n=178, NC=47, CI=131)
  tht-mfi                   OR=2.1476  95%CI (1.6502-2.7948)  p=0.0000 ***
  tht-tlag                  OR=0.8957  95%CI (0.8613-0.9316)  p=0.0000 ***
  tht-T20                   OR=0.8894  95%CI (0.8530-0.9273)  p=0.0000 ***
  tht-t50                   OR=0.8918  95%CI (0.8543-0.9310)  p=0.0000 ***
  forming rate              OR=1.0707  95%CI (1.0055-1.1401)  p=0.0329 *
  dls-peak-number           OR=0.0245  95%CI (0.0084-0.0710)  p=0.0000 ***
  dls-peak-1-intensity      OR=1.1975  95%CI (1.1253-1.2743)  p=0.0000 ***
  dls-peak-2-intensity      OR=0.8785  95%CI (0.8258-0.9347)  p=0.0000 ***
  dls-peak-1-size           OR=1.0026  95%CI (1.0010-1.0042)  p=0.0018 **
  dls-peak-2-size           OR=0.9976  95%CI (0.9959-0.9992)  p=0.0036 **
  dls-peak-1-pd             OR=0.93